# Colab entry notebook — MobileNetV3-Large (Member 4)

Shared Colab workflow per `docs/COLAB.md`. This notebook clones the reviewed, tagged
commit and calls the repository's `run_pipeline.py` / `scripts/prepare_data.py`; it does
**not** duplicate training or data logic. Do not edit code inside Colab — make changes
through normal branches/pull requests, then pull the reviewed commit here.

Research-only: outputs are for the SE4050 coursework comparison, not a diagnostic tool.

Sections:
1. Verify GPU runtime
2. Clone the tagged commit and install dependencies
3. Mount Drive for durable outputs
4. Stage the dataset inside `/content`
5. Configure model/seed
6. Train (resumable) for seeds 42, 123, 2026
7. Confirm frozen BatchNorm + phase-2 optimizer/scheduler rebuild
8. Evaluate validation split and record the experiment handoff


## 1. Verify GPU runtime


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
!nvidia-smi


## 2. Clone the tagged commit

Set `REPO_URL` and `TRAINING_TAG` before running. `TRAINING_TAG` must be the tag created
after section 4.3 of `docs/TEAM_WORKFLOW.md` (e.g. `training-v1.0`); no member should train
from an untagged or unreviewed commit.


In [ ]:
REPO_URL = "https://github.com/YasinduRasangaKarawita/DL-Project.git"
TRAINING_TAG = "training-v1.0"

!git clone {REPO_URL} REPOSITORY
%cd REPOSITORY
!git checkout {TRAINING_TAG}
!git rev-parse HEAD
!pip install -q -r requirements.txt


## 3. Mount Drive for durable outputs

`/content` storage disappears when the runtime ends. All checkpoints, history, and results
must be written straight to Drive so a disconnect never loses completed epochs.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ARTIFACT_ROOT = "/content/drive/MyDrive/SE4050_PlantDisease"
Path(ARTIFACT_ROOT).mkdir(parents=True, exist_ok=True)
print("Artifact root:", ARTIFACT_ROOT)


## 4. Stage the dataset inside `/content`

Reading thousands of small files directly from mounted Drive is slow, so the extracted
color images are staged inside the cloned repository at `data/raw/plantvillage/color`,
not on Drive. The committed manifests under `data/splits/` are frozen; this step must
never regenerate or resplit them, only verify them.


In [ ]:
!python scripts/prepare_data.py download
!python scripts/prepare_data.py validate
!python scripts/prepare_data.py validate-manifests


## 5. Configure model and seeds

`MODEL_NAME` is fixed to `mobilenet_v3` for this notebook (Member 4's assigned model).
`SEEDS` must match the frozen `training.seeds` list in `configs/config.yaml`.


In [ ]:
MODEL_NAME = "mobilenet_v3"
SEEDS = [42, 123, 2026]
CONFIG_PATH = "configs/config.yaml"

import yaml

with open(CONFIG_PATH, encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

assert SEEDS == config["training"]["seeds"], "SEEDS must match configs/config.yaml"
print("initial_epochs:", config["training"]["initial_epochs"])
print("fine_tune_epochs:", config["training"]["fine_tune_epochs"])
print("learning_rate:", config["training"]["learning_rate"])
print("fine_tune_learning_rate:", config["training"]["fine_tune_learning_rate"])


## 6. Train (resumable) for every seed

Each seed writes to its own run directory under `ARTIFACT_ROOT`. The run ID is cached in a
small marker file on Drive so re-running this cell after any disconnect resumes the same
run instead of starting a new one — `train_experiment` is a no-op once every epoch is
already recorded, so it is always safe to re-run.


In [ ]:
import json
from pathlib import Path

from src.training.runner import train_experiment

RUN_ID_DIR = Path(ARTIFACT_ROOT) / "models" / MODEL_NAME / ".run_ids"
RUN_ID_DIR.mkdir(parents=True, exist_ok=True)


def train_or_resume_seed(seed: int) -> dict:
    run_id_file = RUN_ID_DIR / f"seed_{seed}.txt"
    resume_path = None
    run_id = None
    if run_id_file.exists():
        run_id = run_id_file.read_text(encoding="utf-8").strip()
        candidate = Path(ARTIFACT_ROOT) / "models" / MODEL_NAME / run_id / "last_resume.pt"
        if candidate.is_file():
            resume_path = str(candidate)

    record = train_experiment(
        model_name=MODEL_NAME,
        seed=seed,
        config_path=CONFIG_PATH,
        resume_path=resume_path,
        run_id=run_id,
        device_preference="cuda",
        allow_dirty=False,
        artifact_root=ARTIFACT_ROOT,
    )
    run_id_file.write_text(record["run_id"], encoding="utf-8")
    return record


run_records = {}
for seed in SEEDS:
    print(f"--- seed {seed} ---")
    run_records[seed] = train_or_resume_seed(seed)
    print(json.dumps(run_records[seed], indent=2))


## 7. Confirm frozen BatchNorm and phase-2 optimizer/scheduler rebuild

`ModelTrainer._set_frozen_batchnorm_eval` (unit-tested in
`tests/test_training_interface.py::test_trainer_uses_macro_f1_and_keeps_frozen_batchnorm_fixed`)
forces every BatchNorm module whose parameters are not trainable into `eval()` mode before
each epoch, so its running statistics cannot update while frozen. The cell below repeats
that check directly against the real MobileNetV3-Large architecture instead of a toy model.


In [ ]:
import json
import torch

from src.models.registry import build_model
from src.training.trainer import ModelTrainer

class_mapping = json.loads(Path(config["dataset"]["manifest_dir"], "class_mapping.json").read_text(encoding="utf-8"))
num_classes = len(class_mapping["classes"])

frozen_check_config = {MODEL_NAME: dict(config["models"][MODEL_NAME])}
model_spec = build_model(MODEL_NAME, num_classes=num_classes, models_config=frozen_check_config)
model = model_spec.model

frozen_bn_modules = [
    module
    for module in model.modules()
    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm)
    and all(not parameter.requires_grad for parameter in module.parameters(recurse=False))
]
assert frozen_bn_modules, "Expected at least one frozen BatchNorm module while freeze_base=True"
before = frozen_bn_modules[0].running_mean.clone()

trainer = ModelTrainer(model, torch.optim.Adam(model.parameters(), lr=1e-3), device=torch.device("cpu"))
dummy_images = torch.randn(4, 3, 224, 224)
dummy_labels = torch.zeros(4, dtype=torch.long)
loader = [(dummy_images, dummy_labels)]
trainer.train_epoch(loader)

after = frozen_bn_modules[0].running_mean
assert torch.equal(before, after), "Frozen BatchNorm running_mean changed during phase 1"
print("Frozen BatchNorm running_mean unchanged after a phase-1 epoch: OK")


In [ ]:
from src.training.checkpoints import load_checkpoint

for seed, record in run_records.items():
    last_resume_path = record["artifacts"]["last_resume"]["path"]
    checkpoint = load_checkpoint(last_resume_path, torch.device("cpu"))
    schedule = checkpoint["run_metadata"]["schedule"]
    assert schedule == {
        "initial_epochs": config["training"]["initial_epochs"],
        "fine_tune_epochs": config["training"]["fine_tune_epochs"],
    }, f"seed {seed}: schedule does not match the frozen protocol"
    assert checkpoint["phase"] == "fine_tune", f"seed {seed}: training did not reach the fine-tune phase"

    optimizer_lr = checkpoint["optimizer_state_dict"]["param_groups"][0]["lr"]
    expected_lr = config["training"]["fine_tune_learning_rate"]
    assert optimizer_lr == expected_lr, (
        f"seed {seed}: optimizer lr {optimizer_lr} does not match fine_tune_learning_rate {expected_lr}"
    )
    print(f"seed {seed}: phase=fine_tune, optimizer lr={optimizer_lr} matches fine_tune_learning_rate — OK")


## 8. Evaluate validation split and record the experiment handoff

Do not add `--confirm-locked-test` here — the locked test set is evaluated once by
Member 3 after model selection is complete for all four models.


In [ ]:
from src.evaluation.runner import evaluate_checkpoint

for seed, record in run_records.items():
    checkpoint_path = record["artifacts"]["best_inference"]["path"]
    result = evaluate_checkpoint(
        checkpoint_path=checkpoint_path,
        split="validation",
        device_preference="cuda",
        output_root=ARTIFACT_ROOT,
    )
    print(f"seed {seed}: run_id={record['run_id']} best_validation_macro_f1={record['best_validation_macro_f1']:.4f}")
    print(f"  inference checkpoint sha256: {record['artifacts']['best_inference']['sha256']}")
    print(f"  resume checkpoint sha256: {record['artifacts']['last_resume']['sha256']}")


Fill in the handoff template from `docs/TEAM_WORKFLOW.md` (section 16) with the Git SHA,
config path, dataset/manifest checksums, seed, GPU/runtime, best epoch, best validation
macro F1, and both checkpoint paths/hashes printed above before sending results to
Member 4 for release packaging.
